> **Notebook version:** `v0.13 — phase 13 — 2026-04-19`
> If this stamp is older than the current `docs/experiments/` index, reload via *Kernel → Restart & Run All* — the browser may be holding a stale copy.

# 05 — Batch Rhythm Analysis

Analyze a folder of audio files: visual timelines, audio playback, and LLM explanations.

**Setup:**
1. Download samples: `python scripts/download_samples.py`
2. Or drop your own `.wav` / `.mp3` / `.flac` files into `data/songs/`
3. Start Ollama: `ollama serve` (if using local API)

In [ ]:
# ── Configuration ─────────────────────────────────────────────
USE_CLOUD = True
CLOUD_BASE_URL = "http://localhost:11434/v1"  # Ollama default
CLOUD_MODEL_ID = "gemma4:e4b"

# Folder to scan for audio files (relative to notebook)
AUDIO_DIR = "../data/songs"

# How many tracks to analyze (None = all)
MAX_TRACKS = 10

AUDIO_DIR = "../data/songs/eval_set"
MAX_TRACKS = None

# Which questions to ask (None = all)
# Options: "time_signature", "counting", "style_fit", "difficulty", "exercise",
#          "dancer", "song_arc", "sections"
# Legacy alias still accepted in this notebook: "style" -> "style_fit"
QUESTIONS = ["time_signature", "style_fit", "dancer", "song_arc", "sections"]

# Dance-style tagging strategy for analysis:
# - "folder": infer style from first folder under AUDIO_DIR (recommended for eval_set)
# - "fixed": use FIXED_DANCE_STYLE for every track
STYLE_TAG_MODE = "folder"
FIXED_DANCE_STYLE = None  # e.g. "bachata" when STYLE_TAG_MODE == "fixed"

# Run the Gemma-4 audio perception pass to detect vocal language per track.
# This loads a separate local multimodal model, runs one audio call per track,
# then frees the model before the reasoning stage. Requires enough VRAM for
# Gemma 4 E4B multimodal (~5-8 GB). Set to False on laptops without GPU.
USE_TRANSCRIPTION = True
TRANSCRIPTION_MODE = "language_only"  # or "lyrics"

# Vocal-activity source for section-boundary tuning (Phase 9).
# "demucs"  — default; uses htdemucs to separate vocals, per-frame envelope (~80 MB model)
# "gemma"   — experimental; flips main ↔ instrumental and drifts ~8 counts vs demucs on eval set (Phase 12b, docs/experiments/19-...)
# "none"    — skip vocal-aware passes (outro/intro extension become no-ops)
VOCAL_SOURCE = "demucs"


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, HTML

from rytmi import load_audio, analyze
from rytmi.dsp import describe_sections
from rytmi.viz import interactive_timeline
from rytmi.llm import load_model, load_cloud_model, generate, explain_rhythm
from rytmi.prompts import ALL_QUESTIONS, downbeat_confidence_label
from rytmi.transcribe import load_multimodal_model, transcribe_vocals
from rytmi.vocal_activity import default_vocal_activity_source

import rytmi.llm as _llm_mod; _llm_mod.CLOUD_DEBUG = False


In [ ]:
if USE_CLOUD:
    processor, model = load_cloud_model(model_id=CLOUD_MODEL_ID, base_url=CLOUD_BASE_URL)
else:
    processor, model = load_model()

## 1. Find audio files

In [ ]:
audio_dir = Path(AUDIO_DIR)
extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

audio_files = sorted(
    f for f in audio_dir.rglob("*") if f.suffix.lower() in extensions
)

if MAX_TRACKS:
    audio_files = audio_files[:MAX_TRACKS]

print(f"Found {len(audio_files)} audio files in {audio_dir.resolve()}")
for f in audio_files:
    print(f"  {f.relative_to(audio_dir)}")

## 2. Analyze and visualize

For each track: run DSP, show an interactive timeline with audio playback.

- **Orange bold lines** = downbeats (the "one")
- **Green dashed lines** = beats, numbered within each measure
- **Red thin lines** = detected onsets (all note attacks)
- **M1, M2, ...** = measure numbers
- **Red cursor** = playback position (click timeline to seek)

In [ ]:
def infer_dance_style(file_path: Path, root: Path):
    if STYLE_TAG_MODE == "fixed":
        return FIXED_DANCE_STYLE.lower().strip() if FIXED_DANCE_STYLE else None
    if STYLE_TAG_MODE != "folder":
        raise ValueError(f"Unsupported STYLE_TAG_MODE: {STYLE_TAG_MODE!r}")

    rel = file_path.relative_to(root)
    # Expecting layouts like: eval_set/bachata/*.mp3 or eval_set/kizomba/*.mp3
    return rel.parts[0].lower().strip() if len(rel.parts) >= 2 else None


vocal_source = default_vocal_activity_source(prefer=VOCAL_SOURCE)
print(f"[vocal-activity] configured source preference: {VOCAL_SOURCE}")

results = {}
for f in audio_files:
    label = str(f.relative_to(audio_dir))
    dance_style = infer_dance_style(f, audio_dir)
    try:
        audio = load_audio(f)
        try:
            vocal_env = vocal_source.compute(audio)
        except Exception as _e:
            print(f"  {f.name}: vocal-activity FAILED ({_e}); continuing without envelope")
            vocal_env = None
        result = analyze(audio, dance_style=dance_style, vocal_env=vocal_env)
        results[label] = result

        style_suffix = f" [{dance_style}]" if dance_style else ""
        # Interactive timeline with synced audio playback and moving cursor
        display(HTML(interactive_timeline(result, title=f"{label}{style_suffix}")))

    except Exception as e:
        print(f"  {label}: FAILED ({e})")

print(f"Successfully analyzed {len(results)}/{len(audio_files)} files")

## 2c. Section diagnostic table

Print a per-section table for each analyzed track showing the time range in both seconds and phrase/measure coordinates (P#/M#), drift from the phrase grid in beats (with `*` marking boundaries snapped to the grid), and per-section source signals (RMS ratio, onsets-per-beat). Use this to check whether the DSP's section boundaries line up with where you hear the music change, and why each section was detected.


In [ ]:
for label, result in results.items():
    display(Markdown(f"**{label}**"))
    print(describe_sections(result))
    print()


## 2b. Vocal transcription (optional)

If `USE_TRANSCRIPTION` is on, load a local Gemma-4 multimodal model, run one
audio call per track to detect the vocal language, then free the model before
switching to the Ollama reasoning stage. Two-phase on purpose — both models
do not fit in VRAM at once.


In [ ]:
if USE_TRANSCRIPTION:
    import gc
    mm_processor, mm_model = load_multimodal_model()
    try:
        for label, result in results.items():
            try:
                vocals = transcribe_vocals(
                    result.audio, result, mm_processor, mm_model,
                    mode=TRANSCRIPTION_MODE,
                )
                result.vocals = vocals
                print(f"  {label}: {vocals.language}  (clip @ {vocals.clip_start_s:.1f}s)")
            except Exception as e:
                print(f"  {label}: transcription FAILED ({e})")
    finally:
        del mm_processor, mm_model
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        gc.collect()
        try:
            import ctypes
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass
    print(f"Transcription pass complete for {len(results)} tracks")
else:
    print("USE_TRANSCRIPTION=False; skipping perception pass")


## 3. LLM explanations

Ask Gemma about each track — style, time signature, and dancer perspective.

In [ ]:
import gc
import rytmi.llm as _llm_mod; _llm_mod.CLOUD_DEBUG = False

def _trim_process_memory():
    try:
        import ctypes
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


release = globals().get("vocal_source")
if release is not None:
    release_fn = getattr(release, "release", None)
    if callable(release_fn):
        release_fn()
globals().pop("vocal_source", None)
for name in ("audio", "vocal_env", "f", "dance_style", "style_suffix", "mm_processor", "mm_model"):
    globals().pop(name, None)
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
_trim_process_memory()

STYLE_KEY_ALIASES = {"style": "style_fit"}
if QUESTIONS:
    requested_keys = [STYLE_KEY_ALIASES.get(k, k) for k in QUESTIONS]
    requested_keys = list(dict.fromkeys(requested_keys))  # preserve order, remove duplicates
    unknown = [k for k in requested_keys if k not in ALL_QUESTIONS]
    if unknown:
        valid = ", ".join(sorted(ALL_QUESTIONS.keys()))
        raise KeyError(f"Unknown question key(s): {unknown}. Valid keys: {valid}")
    questions = {k: ALL_QUESTIONS[k] for k in requested_keys}
else:
    questions = ALL_QUESTIONS

# Dancer + sections questions need more tokens — per-section coaching is verbose.
LONG_ANSWER_KEYS = {"dancer", "sections"}
LONG_TOKENS = 1536
DEFAULT_TOKENS = 1024

for label, result in results.items():
    display(Markdown(f"---\n### {label}\n"))

    # Build downbeat confidence display
    conf = result.downbeat_confidence
    if conf is not None:
        conf_label = downbeat_confidence_label(conf)
        conf_display = f"**Downbeat \"1\":** {conf_label} ({conf:.2f}) &nbsp; "
    else:
        conf_display = ""

    # Build vocals display
    if result.vocals is not None:
        vocals_display = f"**Vocals:** {result.vocals.language} &nbsp; "
    else:
        vocals_display = ""

    # Build sections summary
    if result.sections:
        sec_parts = [f"{s.label}({s.start_s:.0f}–{s.end_s:.0f}s)" for s in result.sections]
        sections_display = f"**Sections:** {', '.join(sec_parts)} &nbsp; "
    else:
        sections_display = ""

    display(Markdown(
        f"**Tempo:** {result.tempo:.0f} BPM &nbsp; "
        f"**Beats:** {len(result.beats.times)} &nbsp; "
        f"**Onsets:** {len(result.onsets.times)} &nbsp; "
        f"**Duration:** {result.audio.duration:.1f}s &nbsp; "
        + conf_display
        + vocals_display
        + sections_display
    ))

    for qkey, question in questions.items():
        max_tokens = LONG_TOKENS if qkey in LONG_ANSWER_KEYS else DEFAULT_TOKENS
        response = explain_rhythm(
            result, question=question,
            processor=processor, model=model,
            max_new_tokens=max_tokens,
            vocals=result.vocals,
        )
        display(Markdown(f"**{qkey.replace('_', ' ').title()}:** {response}\n"))


## 4. Summary table

In [ ]:
header = (
    "| Track | Tempo | Beats | Onsets | Duration | Downbeat \"1\" | Vocals |\n"
    "|---|---|---|---|---|---|---|\n"
)
rows = []
for label, r in results.items():
    conf = r.downbeat_confidence
    if conf is not None:
        conf_cell = f"{downbeat_confidence_label(conf)} ({conf:.2f})"
    else:
        conf_cell = "\u2014"
    if r.vocals is not None:
        vocals_cell = r.vocals.language
    else:
        vocals_cell = "\u2014"
    rows.append(
        f"| {label} | {r.tempo:.0f} BPM | {len(r.beats.times)} "
        f"| {len(r.onsets.times)} | {r.audio.duration:.1f}s | {conf_cell} | {vocals_cell} |"
    )
display(Markdown(header + "\n".join(rows)))


## API smoke-test — raw response inspector

Run this cell independently to verify the cloud endpoint works and to inspect the raw response structure.
For **thinking models** (`:thinking` suffix) the answer may arrive in `reasoning_content` instead of `content`,
or `max_tokens` may be exhausted by the reasoning trace before any answer is written.


In [ ]:
import os
from openai import OpenAI

# ── Edit these to test any model/endpoint/prompt ──────────────────────────────
_TEST_BASE_URL = None          # None → falls back to RYTMI_API_BASE_URL env var or CLOUD_BASE_URL
_TEST_MODEL_ID = CLOUD_MODEL_ID  # or replace with a literal string
_TEST_PROMPT   = "Reply with exactly one sentence: what is 4/4 time signature?"
_TEST_MAX_TOKENS = 4096        # increase for thinking models (reasoning eats tokens)
# ─────────────────────────────────────────────────────────────────────────────

_base_url = _TEST_BASE_URL or os.environ.get("RYTMI_API_BASE_URL") or CLOUD_BASE_URL
_api_key  = os.environ.get("RYTMI_API_KEY", "not-needed")

print(f"endpoint : {_base_url}")
print(f"model    : {_TEST_MODEL_ID}")
print(f"max_tokens: {_TEST_MAX_TOKENS}")
print()

_client = OpenAI(base_url=_base_url, api_key=_api_key)

# ── Test 1: user-only (no system role) ───────────────────────────────────────
print("=" * 60)
print("TEST 1: user-only message (no system role)")
print("=" * 60)
_resp1 = _client.chat.completions.create(
    model=_TEST_MODEL_ID,
    messages=[{"role": "user", "content": _TEST_PROMPT}],
    max_tokens=_TEST_MAX_TOKENS,
    temperature=0.7,
)
print("finish_reason:", _resp1.choices[0].finish_reason)
print("content:", repr(_resp1.choices[0].message.content))

# ── Test 2: system + user (what _generate_cloud actually sends) ──────────────
print()
print("=" * 60)
print("TEST 2: system + user message (what generate() sends)")
print("=" * 60)
_SYSTEM = (
    "You are a music teacher explaining rhythm concepts. "
    "Be concise and practical. Use musical terminology when helpful. "
    "When discussing rhythm patterns, describe how a musician would count them. "
    "Base your answers only on the analysis data provided — do not invent details."
)
_resp2 = _client.chat.completions.create(
    model=_TEST_MODEL_ID,
    messages=[
        {"role": "system", "content": _SYSTEM},
        {"role": "user", "content": _TEST_PROMPT},
    ],
    max_tokens=_TEST_MAX_TOKENS,
    temperature=0.7,
)
print("finish_reason:", _resp2.choices[0].finish_reason)
print("content:", repr(_resp2.choices[0].message.content))

# ── Test 3: system folded into user turn (Gemma-native workaround) ───────────
print()
print("=" * 60)
print("TEST 3: system folded into user turn (Gemma-native workaround)")
print("=" * 60)
_resp3 = _client.chat.completions.create(
    model=_TEST_MODEL_ID,
    messages=[{"role": "user", "content": f"{_SYSTEM}\n\n{_TEST_PROMPT}"}],
    max_tokens=_TEST_MAX_TOKENS,
    temperature=0.7,
)
print("finish_reason:", _resp3.choices[0].finish_reason)
print("content:", repr(_resp3.choices[0].message.content))
